# SURDS Audit

Source: `bonbon-rj/SURDS` (HuggingFace, cloned via git-lfs)
Location: `/mnt/data4/shasta/amar.amarjyoti/research_data/raw/surds`

**Important finding**: SURDS is NOT a pre-generated VQA dataset. It stores **per-image spatial metadata** (bboxes, 3D coordinates, yaws, distances, categories). VQA questions are generated via prompt templates at train/eval time (see Drive-MLLM repo).

Covers plan sections 0.2 (raw inspection) and 0.3.1 (SURDS-specific deep dive).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from PIL import Image

DATA_DIR = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/raw/surds')
AUDIT_OUT = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/audit/reports/surds')
AUDIT_OUT.mkdir(parents=True, exist_ok=True)
print(f'Data dir: {DATA_DIR}')
print(f'Audit out: {AUDIT_OUT}')

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(DATA_DIR / 'train' / 'metadata.jsonl')
val = load_jsonl(DATA_DIR / 'validation' / 'metadata.jsonl')
print(f'Train: {len(train):,} samples')
print(f'Val:   {len(val):,} samples')
print(f'Total: {len(train) + len(val):,}')

## 0.2.1 Schema Discovery

In [ ]:
# Inspect a few samples to understand the schema
for i, sample in enumerate(train[:2]):
    print(f'--- Sample {i} ---')
    print(json.dumps(sample, indent=2))
    print()

In [16]:
# Field inventory
fields = Counter()
for s in train:
    for k in s.keys():
        fields[k] += 1
print('Train field occurrence counts:')
for k, v in fields.most_common():
    print(f'  {k}: {v}/{len(train)} ({100*v/len(train):.1f}%)')

Train field occurrence counts:
  file_name: 27152/27152 (100.0%)
  descs: 27152/27152 (100.0%)
  xy2Ds: 27152/27152 (100.0%)
  xyz3Ds: 27152/27152 (100.0%)
  bboxes2D: 27152/27152 (100.0%)
  categories: 27152/27152 (100.0%)
  yaws: 27152/27152 (100.0%)
  yaw_descs: 27152/27152 (100.0%)
  depths: 27152/27152 (100.0%)
  distances: 27152/27152 (100.0%)


## 0.2.2 Basic Counts

In [17]:
# Objects per sample (each sample = one image with N object annotations)
train_obj_counts = [len(s['descs']) for s in train]
val_obj_counts = [len(s['descs']) for s in val]

print(f'Train objects/sample: mean={np.mean(train_obj_counts):.2f}, median={np.median(train_obj_counts):.0f}, min={min(train_obj_counts)}, max={max(train_obj_counts)}')
print(f'Val   objects/sample: mean={np.mean(val_obj_counts):.2f}, median={np.median(val_obj_counts):.0f}, min={min(val_obj_counts)}, max={max(val_obj_counts)}')
print(f'Total objects in train: {sum(train_obj_counts):,}')
print(f'Total objects in val:   {sum(val_obj_counts):,}')

Train objects/sample: mean=1.14, median=1, min=1, max=4
Val   objects/sample: mean=1.14, median=1, min=1, max=4
Total objects in train: 31,031
Total objects in val:   6,773


In [ ]:
# Camera distribution
def get_cam(s):
    return s['file_name'].split('/')[1]

cam_train = Counter(get_cam(s) for s in train)
cam_val = Counter(get_cam(s) for s in val)

cam_df = pd.DataFrame({'train': cam_train, 'val': cam_val}).fillna(0).astype(int)
cam_df.loc['total'] = cam_df.sum()
cam_df

## 0.2.3 Task Type Analysis

SURDS QA is template-generated from metadata, not pre-stored. The metadata supports these task families from our taxonomy:
- `spatial_relation_geometry` ← primary (yaw, depth, distance, 3D position)
- `object_scene_perception` ← secondary (category, description)
- `regional_perception` ← 2D bbox + description

In [ ]:
# Object category distribution
all_cats = Counter()
for s in train:
    for c in s['categories']:
        all_cats[c] += 1

print('Top categories (train):')
for c, n in all_cats.most_common(20):
    print(f'  {c}: {n:,}')

In [ ]:
# Yaw direction distribution
all_yaws = Counter()
for s in train:
    for y in s['yaw_descs']:
        all_yaws[y] += 1
print('Yaw direction distribution:')
for y, n in all_yaws.most_common():
    print(f'  {y}: {n:,}')

## 0.2.4 Answer Analysis

Since SURDS has no pre-made answers, this section looks at **ground-truth target fields** (numerical values that templates would turn into answers).

In [ ]:
# Distance distribution (a key answer target)
all_distances = [d for s in train for d in s['distances']]
all_depths = [d for s in train for d in s['depths']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_distances, bins=50)
axes[0].set_title(f'Distance distribution (n={len(all_distances):,})')
axes[0].set_xlabel('distance (m)')
axes[1].hist(all_depths, bins=50)
axes[1].set_title(f'Depth distribution')
axes[1].set_xlabel('depth (m)')
plt.tight_layout()
plt.savefig(AUDIT_OUT / 'distance_depth_hist.png', dpi=100)
plt.show()

print(f'Distance: mean={np.mean(all_distances):.1f}, median={np.median(all_distances):.1f}, p95={np.percentile(all_distances, 95):.1f}')

## 0.2.5 Image Analysis

In [ ]:
# Sample a few images for resolution check
import random
random.seed(42)
sample_paths = random.sample(train, 50)

resolutions = []
broken = 0
for s in sample_paths:
    img_path = DATA_DIR / 'train' / s['file_name']
    try:
        with Image.open(img_path) as im:
            resolutions.append(im.size)
    except Exception as e:
        broken += 1

res_counter = Counter(resolutions)
print(f'Resolution distribution (50 random samples):')
for res, n in res_counter.most_common():
    print(f'  {res}: {n}')
print(f'Broken: {broken}')

### Visualize Sample Images with Annotations

Display random samples with 2D bboxes, descriptions, and spatial metadata overlaid.

In [ ]:
import random
import matplotlib.patches as patches

def visualize_sample(sample, split='train', ax=None):
    img_path = DATA_DIR / split / sample['file_name']
    img = Image.open(img_path)
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    for bbox, desc, cat, dist, yaw in zip(sample['bboxes2D'], sample['descs'], sample['categories'], sample['distances'], sample['yaw_descs']):
        x0, y0, x1, y1 = bbox
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        label = f'{desc}\n{cat} | {dist}m {yaw}'
        ax.text(x0, max(y0-5, 10), label, color='white', fontsize=8, bbox=dict(facecolor='black', alpha=0.6, pad=2))
    cam = sample['file_name'].split('/')[1]
    ax.set_title(f'{cam} | {Path(sample["file_name"]).name}', fontsize=9)
    ax.axis('off')
    return ax

random.seed(0)
viz_samples = random.sample(train, 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for sample, ax in zip(viz_samples, axes.flat):
    visualize_sample(sample, split='train', ax=ax)
plt.tight_layout()
plt.savefig(AUDIT_OUT / 'sample_visualizations.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# One detailed sample with full metadata printout alongside image
detailed = random.choice(train)
fig, ax = plt.subplots(figsize=(14, 7))
visualize_sample(detailed, split='train', ax=ax)
plt.show()
print('Full metadata:')
print(json.dumps(detailed, indent=2))

## 0.2.6 Quality Red Flags

In [ ]:
# Check for empty annotations, duplicate file_names, etc.
empty_descs = sum(1 for s in train if not s['descs'])
empty_bboxes = sum(1 for s in train if not s['bboxes2D'])

file_names = [s['file_name'] for s in train]
dup_files = len(file_names) - len(set(file_names))

print(f'Empty descs: {empty_descs}')
print(f'Empty bboxes: {empty_bboxes}')
print(f'Duplicate file_names: {dup_files}')

# Length consistency (descs, xy2Ds, bboxes2D should all match per sample)
inconsistent = 0
for s in train:
    ln = len(s['descs'])
    for k in ['xy2Ds', 'xyz3Ds', 'bboxes2D', 'categories', 'yaws', 'yaw_descs', 'depths', 'distances']:
        if len(s[k]) != ln:
            inconsistent += 1
            break
print(f'Length-inconsistent samples: {inconsistent}')

## 0.3.1 SURDS Deep Dive — Spatial Metadata

In [ ]:
# 3D position distribution (ego-centric)
xs = [xyz[0] for s in train for xyz in s['xyz3Ds']]
ys = [xyz[1] for s in train for xyz in s['xyz3Ds']]
zs = [xyz[2] for s in train for xyz in s['xyz3Ds']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(xs, bins=50); axes[0].set_title('X (lateral, m)')
axes[1].hist(ys, bins=50); axes[1].set_title('Y (vertical, m)')
axes[2].hist(zs, bins=50); axes[2].set_title('Z (forward, m)')
plt.tight_layout()
plt.savefig(AUDIT_OUT / 'xyz3d_hist.png', dpi=100)
plt.show()

In [ ]:
# Derive difficulty heuristic: far-away objects with narrow bboxes are harder
# Occlusion-like: overlapping bboxes in same image
difficulties = []
for s in train:
    for d in s['distances']:
        difficulties.append(d)

# Difficulty buckets: near (<10m), mid (10-25m), far (>25m)
near = sum(1 for d in difficulties if d < 10)
mid = sum(1 for d in difficulties if 10 <= d <= 25)
far = sum(1 for d in difficulties if d > 25)
total = len(difficulties)
print(f'Difficulty buckets (by distance):')
print(f'  Near (<10m):  {near:,} ({100*near/total:.1f}%)')
print(f'  Mid (10-25m): {mid:,} ({100*mid/total:.1f}%)')
print(f'  Far (>25m):   {far:,} ({100*far/total:.1f}%)')

## SURDS QA Generation from Metadata

SURDS generates 6 types of VQA from the spatial metadata using templates (`Drive-MLLM/prompt/prompts_reasoning/`):

| Type | Question | Metadata Fields | Multi-object? |
|------|----------|----------------|---------------|
| `yaw` | Which direction is X facing? | `yaw_descs` | No |
| `xy2d` | Where is X located? | `xy2Ds` | No |
| `depth` | How far is X from camera? | `depths` | No |
| `distance` | Which is closer/farther? | `distances` | Yes (2+ objects) |
| `left-right` | Which is further left/right? | `xy2Ds` | Yes |
| `front-back` | Is X in front of/behind Y? | `depths` | Yes |

Below we generate example QA pairs for each type and visualize them with their source image.

**CoT traces** are NOT included in the released data — must be generated via `Drive-MLLM/training/sft/gen_cot.py` against Qwen2.5-VL-72B-Instruct. That's a future pipeline step.

In [ ]:
import random
import matplotlib.patches as patches

random.seed(42)

# Camera facing directions (from Drive-MLLM convention)
CAM_FACING = {
    'CAM_FRONT': 'North', 'CAM_FRONT_LEFT': 'Northwest', 'CAM_FRONT_RIGHT': 'Northeast',
    'CAM_BACK': 'South', 'CAM_BACK_LEFT': 'Southwest', 'CAM_BACK_RIGHT': 'Southeast',
}

# Load prompt templates
TEMPLATE_DIR = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/Drive-MLLM/prompt/prompts_reasoning')
templates = {}
for f in sorted(TEMPLATE_DIR.glob('*.txt')):
    templates[f.stem] = f.read_text()
print('Templates loaded:', list(templates.keys()))

# ── Helper: generate all 6 QA types for a sample ──
def generate_qa_for_sample(s):
    """Generate example QA pairs from one metadata sample."""
    cam = s['file_name'].split('/')[1]
    cam_dir = CAM_FACING.get(cam, 'North')
    qas = []
    
    # Single-object tasks (pick first object)
    obj = 'the ' + s['descs'][0]
    
    # 1. Yaw VQA: Which direction is X facing?
    correct_yaw = s['yaw_descs'][0]
    all_dirs = ['North', 'South', 'East', 'West', 'Northeast', 'Northwest', 'Southeast', 'Southwest']
    distractors = [d for d in all_dirs if d != correct_yaw]
    options = random.sample(distractors, 3) + [correct_yaw]
    random.shuffle(options)
    qas.append({
        'type': 'yaw',
        'Q': f'Which direction is {obj} facing in the image? (Camera facing {cam_dir})',
        'options': options,
        'A': correct_yaw,
    })
    
    # 2. XY2D VQA: Where is X located?
    xy = s['xy2Ds'][0]
    qas.append({
        'type': 'xy2d',
        'Q': f'Where is {obj} located in the image?',
        'options': None,
        'A': f'[{xy[0]}, {xy[1]}]',
    })
    
    # 3. Depth VQA: How far is X from camera?
    depth = s['depths'][0]
    # Generate answer range + 2 distractors (simplified from Drive-MLLM logic)
    half = 3
    ans_range = [max(1, round(depth - half)), round(depth + half)]
    d2 = [ans_range[1] + 2, ans_range[1] + 6]
    d3 = [max(1, ans_range[0] - 6), max(1, ans_range[0] - 2)]
    fmt = lambda r: f'Between {r[0]} {"meter" if r[0]<=1 else "meters"} and {r[1]} {"meter" if r[1]<=1 else "meters"}'
    qas.append({
        'type': 'depth',
        'Q': f'How far is the vertical distance of {obj} from the camera?',
        'options': [fmt(ans_range), fmt(d2), fmt(d3)],
        'A': fmt(ans_range),
    })
    
    # Multi-object tasks (need 2+ objects)
    if len(s['descs']) >= 2:
        obj1 = 'the ' + s['descs'][0]
        obj2 = 'the ' + s['descs'][1]
        
        # 4. Distance VQA: Which is closer?
        d1, d2_val = s['distances'][0], s['distances'][1]
        if abs(d1 - d2_val) <= 1:
            dis_answer = 'Almost the same'
        elif d1 < d2_val:
            dis_answer = obj1.capitalize()
        else:
            dis_answer = obj2.capitalize()
        qas.append({
            'type': 'distance',
            'Q': f'Which object, {obj1} or {obj2}, is closer to the camera?',
            'options': [obj1.capitalize(), obj2.capitalize(), 'Almost the same'],
            'A': dis_answer,
        })
        
        # 5. Left-Right VQA
        x1, x2 = s['xy2Ds'][0][0], s['xy2Ds'][1][0]
        if abs(x1 - x2) < 100:
            lr_answer = 'Almost the same'
        elif x1 < x2:
            lr_answer = obj1.capitalize()
        else:
            lr_answer = obj2.capitalize()
        qas.append({
            'type': 'left-right',
            'Q': f'Which is further left, {obj1} or {obj2}?',
            'options': [obj1.capitalize(), obj2.capitalize(), 'Almost the same'],
            'A': lr_answer,
        })
        
        # 6. Front-Back VQA
        dp1, dp2 = s['depths'][0], s['depths'][1]
        if abs(dp1 - dp2) < 0.5:
            fb_answer = 'Almost the same in terms of front-back position'
        elif dp1 > dp2:
            fb_answer = 'Yes'
        else:
            fb_answer = 'No'
        qas.append({
            'type': 'front-back',
            'Q': f'Is {obj1} in front of {obj2}?',
            'options': ['Yes', 'No', 'Almost the same in terms of front-back position'],
            'A': fb_answer,
        })
    
    return qas

# ── Generate and display for a multi-object sample ──
multi_obj = [s for s in train if len(s['descs']) >= 2]
print(f'Multi-object samples: {len(multi_obj):,} / {len(train):,} ({100*len(multi_obj)/len(train):.1f}%)')

sample = random.choice(multi_obj)
qas = generate_qa_for_sample(sample)

# Show image
fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 1.2]})
visualize_sample(sample, split='train', ax=ax_img)

# Show generated QA
text_lines = []
for qa in qas:
    text_lines.append(f"[{qa['type'].upper()}]")
    text_lines.append(f"Q: {qa['Q']}")
    if qa['options']:
        for i, opt in enumerate(qa['options']):
            marker = '→' if opt == qa['A'] else ' '
            text_lines.append(f"  {marker} {opt}")
    else:
        text_lines.append(f"  A: {qa['A']}")
    text_lines.append('')

ax_text.text(0.02, 0.98, '\n'.join(text_lines), fontsize=9, va='top', family='monospace',
             transform=ax_text.transAxes)
ax_text.axis('off')
ax_text.set_title('Generated QA from metadata', fontsize=11)
plt.tight_layout()
plt.savefig(AUDIT_OUT / 'qa_generation_example.png', dpi=100, bbox_inches='tight')
plt.show()

### Visualize Generated QA (one per template type)

Loads the Phase-1 canonical QA JSONL produced by `examples/data_scripts/surds_gen_qa.py` and shows one real example per template: `yaw`, `xy2d`, `depth`, `distance`, `lr`, `fb`. Each panel: source image with referenced object bboxes + prompt/options/gold answer.

In [ ]:
# Load generated QA (Phase 1 output)
QA_JSONL = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/processed/surds/train_qa.jsonl')
qa_all = load_jsonl(QA_JSONL)
print(f'Loaded {len(qa_all):,} QAs from {QA_JSONL.name}')

# Index raw metadata by file_name so we can recover bboxes for referenced objects.
meta_by_path = {str(DATA_DIR / 'train' / s['file_name']): s for s in train}

# Pick one example per template type.
random.seed(3)
by_type = {}
for q in qa_all:
    by_type.setdefault(q['template_type'], []).append(q)
picks = {t: random.choice(by_type[t]) for t in ['yaw', 'xy2d', 'depth', 'distance', 'lr', 'fb']}

def bboxes_for_qa(qa):
    """Return list of (bbox, label) for objects referenced in this QA."""
    meta = meta_by_path.get(qa['image_path'])
    if meta is None:
        return []
    md = qa['metadata']
    # single-obj: metadata carries obj_bbox directly
    if 'obj_bbox' in md:
        return [(md['obj_bbox'], md['obj'])]
    # multi-obj: look up both named objects in the per-image metadata
    out = []
    for key in ('obj1', 'obj2'):
        name = md.get(key)
        if name in meta['descs']:
            idx = meta['descs'].index(name)
            out.append((meta['bboxes2D'][idx], name))
    return out

def render_qa_panel(qa, ax_img, ax_text):
    img = Image.open(qa['image_path'])
    ax_img.imshow(img)
    colors = ['lime', 'cyan']
    for (bbox, label), c in zip(bboxes_for_qa(qa), colors):
        x0, y0, x1, y1 = bbox
        ax_img.add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0,
                                           linewidth=2, edgecolor=c, facecolor='none'))
        ax_img.text(x0, max(y0-5, 10), label, color='white', fontsize=8,
                    bbox=dict(facecolor=c, alpha=0.7, pad=2))
    cam = Path(qa['image_path']).parent.name
    ax_img.set_title(f"[{qa['template_type']}] {cam}", fontsize=10)
    ax_img.axis('off')

    # Condensed prompt: strip the fixed <think>/<answer> boilerplate for readability.
    prompt = qa['prompt'].split('Reason carefully')[0].strip()
    lines = [prompt, '']
    if qa['options']:
        lines.append('Options:')
        for opt in qa['options']:
            marker = '>' if opt == qa['answer'] else ' '
            lines.append(f'  {marker} {opt}')
        lines.append('')
    lines.append(f"GOLD: {qa['answer']}")
    ax_text.text(0.0, 1.0, '\n'.join(lines), fontsize=8.5, va='top',
                 family='monospace', transform=ax_text.transAxes, wrap=True)
    ax_text.axis('off')

order = ['yaw', 'xy2d', 'depth', 'distance', 'lr', 'fb']
fig, axes = plt.subplots(len(order), 2, figsize=(18, 5 * len(order)),
                         gridspec_kw={'width_ratios': [1.3, 1]})
for row, t in zip(axes, order):
    render_qa_panel(picks[t], row[0], row[1])
plt.tight_layout()
plt.savefig(AUDIT_OUT / 'generated_qa_per_template.png', dpi=100, bbox_inches='tight')
plt.show()

## Deliverables

In [ ]:
# Summary stats JSON
summary = {
    'dataset': 'SURDS',
    'source': 'bonbon-rj/SURDS',
    'format': 'per-image metadata (object annotations); QA generated via templates',
    'counts': {
        'train_samples': len(train),
        'val_samples': len(val),
        'train_objects': sum(train_obj_counts),
        'val_objects': sum(val_obj_counts),
    },
    'cameras': dict(cam_train),
    'top_categories': dict(all_cats.most_common(10)),
    'yaw_distribution': dict(all_yaws),
    'distance_stats': {
        'mean': float(np.mean(all_distances)),
        'median': float(np.median(all_distances)),
        'p95': float(np.percentile(all_distances, 95)),
    },
    'quality_flags': {
        'empty_descs': empty_descs,
        'duplicate_filenames': dup_files,
        'length_inconsistent': inconsistent,
    },
    'task_families_supported': ['spatial_relation_geometry', 'object_scene_perception', 'regional_perception'],
}

with open(AUDIT_OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:', AUDIT_OUT / 'summary.json')
print(json.dumps(summary, indent=2))

In [18]:
# 10 representative examples (diverse across cameras/categories)
import random
random.seed(0)
examples = random.sample(train, 10)
with open(AUDIT_OUT / 'examples.json', 'w') as f:
    json.dump(examples, f, indent=2)
print('Saved 10 examples to:', AUDIT_OUT / 'examples.json')

Saved 10 examples to: /mnt/data4/shasta/amar.amarjyoti/research_data/audit/reports/surds/examples.json
